# Integer and fixed-point register conversions

Copyright 2024 Google LLC. Adapted under Apache-2.0 from Qualtran v0.7.0, commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable, Hashable
from dataclasses import dataclass
from html import escape
from typing import cast

import torch
from IPython import display as notebook_display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum import Circuit
from flagquantum.algorithms.first_quantization_resource_model import (
    _signed_conversion_cost,
)
from flagquantum.compiler._component_trace.call_graph import (
    SymbolicCallGraph,
    symbolic_call_graph,
)
from flagquantum.compiler._component_trace.resource_generalization import (
    FanoutCall,
    FixedTCall,
    ResourceCall,
)
from flagquantum.compiler.registers.bits.sign_extension import sign_extend_ir
from flagquantum.compiler.registers.sorting.contiguous_index import (
    source_contiguous_index_cost,
)
from flagquantum.core.ir.program.register_types import (
    fixed_point_register_type,
    ones_complement_register_type,
    signed_register_type,
    unsigned_register_type,
)
from flagquantum.core.ir.program.registers import Register, RegisterSide, RegisterType
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.register_graph_table import signature_instance_table
from flagquantum.drawer.register_score import RegisterStep

# Optional IPython functions enter through the presentation boundary.
SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


@dataclass(frozen=True)
class ConversionExample:
    name: str
    signature: Signature
    child: ResourceCall
    multiplicity: int


def sided_signature(inp: RegisterType, out: RegisterType) -> Signature:
    return Signature(
        (
            Register("x", inp, side=RegisterSide.LEFT),
            Register("y", out, side=RegisterSide.RIGHT),
        )
    )


def signature_svg(example: ConversionExample) -> bytes:
    signature = example.signature
    table = signature_instance_table(
        signature,
        RegisterStep(
            example.name,
            tuple(r.name for r in signature.lefts()),
            tuple(r.name for r in signature.rights()),
        ),
        {r.name: f"p{i}" for i, r in enumerate(signature)},
    )
    metadata = ", ".join(f"{r.name}={r.dtype.encoding}[{r.bitsize}]" for r in signature)
    table = table.replace(
        "</TABLE>", f'<TR><TD COLSPAN="2">{escape(metadata)}</TD></TR></TABLE>'
    )
    return render_call_graph_svg(
        "digraph signature { n [shape=plain,label=<" + table + ">]; }"
    )


def call_title(call: Hashable) -> str:
    if isinstance(call, ConversionExample):
        return call.name
    if isinstance(call, FanoutCall):
        return f"Sign fanout ({call.width} targets)"
    if isinstance(call, FixedTCall):
        return "Toffoli"
    return str(call)


def show_graph(example: ConversionExample) -> SymbolicCallGraph[Hashable]:
    root: Hashable = example
    graph: SymbolicCallGraph[Hashable] = symbolic_call_graph(
        root, {root: {example.child: example.multiplicity}}, max_depth=1
    )
    display(
        SVG(
            render_call_graph_svg(
                export_call_graph_dot(graph.root, graph.direct_calls, title=call_title)
            )
        )
    )
    display(graph.leaf_counts)
    return graph


## Ones-complement to twos-complement conversion
Convert a register storing the signed integer representation to two's complement inplace.

#### Parameters
 - `bitsize`: size of the register. 

#### Registers
 - `x`: input signed integer (ones' complement) register.
 - `y`: output signed integer register in two's complement. 

#### References
 - [Fault-Tolerant Quantum Simulations of Chemistry in First Quantization](https://arxiv.org/abs/2105.12767). page 24, 4th paragraph from the bottom.


In [ ]:
# One's-complement input and two's-complement output have distinct metadata.


### Example Instances

In [ ]:
signed_to_twos = ConversionExample(
    "Ones to twos complement",
    sided_signature(ones_complement_register_type(10), signed_register_type(10)),
    FixedTCall("toffoli"),
    _signed_conversion_cost(10).gates.toffoli,
)


#### Graphical Signature

In [ ]:
display(SVG(signature_svg(signed_to_twos)))


### Call Graph

In [ ]:
signed_to_twos_graph = show_graph(signed_to_twos)


In [ ]:
# The index source declares its opaque resource model independently of execution.


### Example Instances

In [ ]:
to_contg_index = ConversionExample(
    "Triangular index",
    Signature(
        (
            Register("mu", unsigned_register_type(4)),
            Register("nu", unsigned_register_type(4)),
            Register("s", unsigned_register_type(8)),
        )
    ),
    FixedTCall("toffoli"),
    source_contiguous_index_cost(4, 8).gates.toffoli,
)


#### Graphical Signature

In [ ]:
display(SVG(signature_svg(to_contg_index)))


### Call Graph

In [ ]:
to_contg_index_graph = show_graph(to_contg_index)


## Sign extension
Sign-Extend a value to a value of larger bitsize.

Useful to implement arithmetic operations with differing operand bitsizes.
A sign extension copies the MSB into the new bits of the wider value. For
example: a 4-bit to 6-bit sign-extension of `1010` gives `111010`.

See sign truncation for the adjoint operation.

#### Parameters
 - `inp_dtype`: input data type.
 - `out_dtype`: output data type. must be same class as `inp_dtype`, and have larger bitsize. 

#### Registers
 - `x`: the input register of type `inp_dtype`
 - `y`: the output register of type `out_dtype`


In [ ]:
# The same fanout owner handles integer and fixed-point sign bits.


### Example Instances

In [ ]:
sign_extend = ConversionExample(
    "Sign extension",
    sided_signature(signed_register_type(8), signed_register_type(16)),
    FanoutCall(8),
    1,
)


In [ ]:
sign_extend_fxp = ConversionExample(
    "Fixed-point sign extension",
    sided_signature(
        fixed_point_register_type(8, 4, signed=True),
        fixed_point_register_type(16, 4, signed=True),
    ),
    FanoutCall(8),
    1,
)


#### Graphical Signature

In [ ]:
display(SVG(signature_svg(sign_extend)))


### Call Graph

In [ ]:
sign_extend_graph = show_graph(sign_extend)


## Sign truncation
Truncate a signed value to a smaller bitsize.

Useful to implement arithmetic operations with differing operand bitsizes.
A signed truncation xors the MSB (sign bit) into the bits to drop, and
deallocates them.

See sign extension for the adjoint operation.


#### Parameters
 - `inp_dtype`: input data type.
 - `out_dtype`: output data type. must be same class as `inp_dtype`, and have smaller bitsize. 

#### Registers
 - `x`: the input register of type `inp_dtype`
 - `y`: the output register of type `out_dtype`


In [ ]:
# Truncation requires a correctly redundant sign prefix before it is released.


### Example Instances

In [ ]:
sign_truncate = ConversionExample(
    "Sign truncation",
    sided_signature(signed_register_type(16), signed_register_type(8)),
    FanoutCall(8),
    1,
)


#### Graphical Signature

In [ ]:
display(SVG(signature_svg(sign_truncate)))


### Call Graph

In [ ]:
sign_truncate_graph = show_graph(sign_truncate)


## Coherent sign extension and clean truncation

An explicit eight-qubit prefix is initially zero and becomes part of the 16-bit output. Truncation restores it to zero. A spectator and every eight-bit input have nonzero complex amplitudes. The same bit action supports signed fixed-point values with four fractional bits, preserving that scale. The original notebook declares this fixed-point example even though its graph is not displayed.

The source conversion and index roots declare Toffoli models without a decomposition. Their native execution evidence and source gaps remain in the complete file audits. Measured fanout gates and one-level model calls are reported separately below.


In [ ]:
word = tuple(range(8))
prefix = tuple(range(8, 16))
gates = sign_extend_ir(word, prefix)
coefficients = torch.arange(512, dtype=torch.float64)
amplitudes = torch.complex(coefficients + 1, 3 * coefficients - 5)
amplitudes /= amplitudes.norm()
initial = torch.zeros(1 << 17, dtype=torch.complex128)
expected = torch.zeros_like(initial)
for raw, amplitude in enumerate(amplitudes):
    value, spectator = raw >> 1, raw & 1
    initial[(value << 9) | spectator] = amplitude
    high = 255 if value >= 128 else 0
    expected[(value << 9) | (high << 1) | spectator] = amplitude
circuit = Circuit(17, inputs=initial, dtype=torch.complex128)
operation = FieldOperation(17, gates)
circuit.append(operation)
torch.testing.assert_close(circuit.state().flatten(), expected, rtol=0, atol=1e-12)
circuit.append(operation)
restored = circuit.state().flatten()
torch.testing.assert_close(restored, initial, rtol=0, atol=1e-12)
assert torch.count_nonzero(restored.reshape(256, 256, 2)[:, 1:, :]) == 0
measured_primitives = dict(Counter(g.name for g in gates))
display(
    {
        "measured_extension_primitives": measured_primitives,
        "modeled_direct_fanout_calls": sign_extend.multiplicity,
    }
)
